In [ ]:
import sys
sys.path.append("..")
from unsloth import FastLanguageModel
from evaluation.metrics import calculate_absa_metrics
import pandas as pd
from tqdm import tqdm
import torch
from ..src.templates import USER_PROMPT

In [ ]:
# 此时加载的是 Base Model + Adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="../outputs/qwen3_cot_finetuned", # 加载刚才保存的 Adapter
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

In [ ]:
def run_inference(text, aspect):
    prompt = USER_PROMPT.format(text=text, aspect=aspect)
    messages = [
        {"role": "user", "content": prompt}
    ]
    inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
    
    outputs = model.generate(inputs, max_new_tokens=128, temperature=0.6)
    return tokenizer.decode(outputs, skip_special_tokens=True)

In [ ]:
df_rest_test = pd.read_json("../data/processed/test_rest_clean.jsonl", lines=True)
df_lap_test = pd.read_json("../data/processed/test_lap_clean.jsonl", lines=True)
df_test = pd.concat([df_rest_test, df_lap_test], ignore_index=True)
# 快速测试: df_test = df_test.head(50)

predictions = []
golds = []

print("Running Evaluation...")
for _, row in tqdm(df_test.iterrows(), total=len(df_test)):
    raw_res = run_inference(row['text'], row)
    
    # 解析输出，提取最后的 label
    # 假设格式是 "... Final Sentiment: positive"
    try:
        pred_label = raw_res.split("Final Sentiment:")[-1].strip().lower()
    except:
        pred_label = "neutral" # 兜底
        
    predictions.append(pred_label)
    golds.append(row['polarity'])

In [ ]:
results = calculate_absa_metrics(golds, predictions)
print("Evaluation Results:")
print(results)

In [ ]:
df_test['predicted'] = predictions
df_test['raw_output'] = ["" for _ in range(len(df_test))] # 可选：保存完整 CoT
df_test.to_csv("../outputs/evaluation_results.csv", index=False)